In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./data/output_from_KNIME/hit_substructure.csv")
print(df.shape)
df.head()

In [ ]:
'''
df= (
    df.groupby("type", group_keys=False)
      .apply(lambda x: x.sample(frac=0.1, random_state=42))
)
'''
df.shape


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw

df.columns = df.columns.str.replace(" ", "")

# フィルタリング: "All"行と、Allのヒット率より高い行を抽出
df_all = df[df["type"] == "All"].iloc[0]
threshold = float(df_all["hit_rate(%)"])
df_filtered = df[(df["hit_rate(%)"] > threshold) | (df["type"] == "All")].copy()
df_filtered = df_filtered.sort_values("hit_rate(%)", ascending=False).reset_index(drop=True)

print(f"Threshold (All hit rate): {threshold:.4f}%")
print(f"Filtered rows: {len(df_filtered)}")

# SMARTS → PIL画像
def smarts_to_pil(smarts):
    if not isinstance(smarts, str) or smarts.strip() == "":
        return None
    mol = Chem.MolFromSmarts(smarts)
    if mol:
        return Draw.MolToImage(mol, size=(150, 150))
    return None

# 図の作成
n_rows = len(df_filtered)
fig, axes = plt.subplots(n_rows, 3, figsize=(10, n_rows * 1.5), 
                          gridspec_kw={'width_ratios': [2, 1, 1]})

for i, (_, row) in enumerate(df_filtered.iterrows()):
    is_all = row["type"] == "All"
    fontweight = "bold" if is_all else "normal"
    bgcolor = "lightyellow" if is_all else "white"
    
    # 構造画像
    axes[i, 0].set_facecolor(bgcolor)
    img = smarts_to_pil(row["substructure"])
    if img:
        axes[i, 0].imshow(img)
    axes[i, 0].axis('off')
    
    # ヒット率
    axes[i, 1].set_facecolor(bgcolor)
    axes[i, 1].text(0.5, 0.5, f'{row["hit_rate(%)"]:.2f}%', 
                    ha='center', va='center', fontsize=12, fontweight=fontweight)
    axes[i, 1].axis('off')
    
    # 95% CI
    axes[i, 2].set_facecolor(bgcolor)
    axes[i, 2].text(0.5, 0.5, str(row["95%CI"]), 
                    ha='center', va='center', fontsize=10, fontweight=fontweight)
    axes[i, 2].axis('off')

# カラムヘッダー
fig.text(0.25, 0.97, "Structure", ha='center', fontsize=14, fontweight='bold')
fig.text(0.55, 0.97, "Hit Rate", ha='center', fontsize=14, fontweight='bold')
fig.text(0.8, 0.97, "95% CI", ha='center', fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("hit_table.png", dpi=300, bbox_inches='tight')
plt.show()